# Chapter 6: Statistical Machine Learning

**Author:** Ruben  
**Date:** September 29, 2026  
**Source:** Practical Statistics for Data Scientists (O'Reilly)

---

## Overview

Machine learning methods for prediction:
- K-Nearest Neighbors (KNN)
- Decision trees
- Bagging and Random Forests
- Boosting (XGBoost)
- Regularization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
sns.set_style('whitegrid')

## 6.1 K-Nearest Neighbors (KNN)

### Theory

**KNN:** Non-parametric, instance-based learning

**Algorithm:**
1. Store all training data
2. For new point, find k nearest neighbors
3. Classification: majority vote
4. Regression: average of neighbors

**Distance metrics:**
- Euclidean: sqrt(sum((x_i - y_i)^2))
- Manhattan: sum(|x_i - y_i|)

**Choosing k:**
- Small k: sensitive to noise
- Large k: smoother boundaries
- Typically k=3,5,7 (odd for binary)

**Preprocessing:**
- Scale features (distance-based)
- Remove irrelevant features

In [ ]:
# KNN example
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                          n_informative=2, n_clusters_per_class=1, 
                          random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features (important for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Try different k values
k_values = [1, 3, 5, 10, 20]
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    acc = knn.score(X_test_scaled, y_test)
    accuracies.append(acc)
    print(f'k={k:2d}: Accuracy = {acc:.3f}')

# Visualize
plt.figure(figsize=(10, 5))
plt.plot(k_values, accuracies, 'o-', linewidth=2, markersize=8)
plt.xlabel('k (number of neighbors)')
plt.ylabel('Test Accuracy')
plt.title('KNN Performance vs k')
plt.grid(True, alpha=0.3)
plt.show()

best_k = k_values[np.argmax(accuracies)]
print(f'\nBest k = {best_k} with accuracy {max(accuracies):.3f}')

## 6.2 Decision Trees

### Theory

**Decision Tree:** Hierarchical if-then rules

**Algorithm (Recursive Partitioning):**
1. Start with all data
2. Find best split (feature + threshold)
3. Split data into two groups
4. Repeat for each group
5. Stop when: max depth, min samples, pure node

**Splitting criteria:**
- **Gini impurity**: sum(p_i * (1 - p_i))
- **Entropy**: -sum(p_i * log(p_i))
- Lower = more pure

**Advantages:**
- Interpretable
- No scaling needed
- Handles non-linear relationships

**Disadvantages:**
- Overfits easily
- High variance
- Solution: ensemble methods

In [ ]:
# Decision tree example
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

train_acc = tree.score(X_train, y_train)
test_acc = tree.score(X_test, y_test)

print(f'Decision Tree Results:')
print(f'Train Accuracy: {train_acc:.3f}')
print(f'Test Accuracy: {test_acc:.3f}')
print(f'Tree Depth: {tree.get_depth()}')
print(f'Number of Leaves: {tree.get_n_leaves()}')

# Visualize tree
plt.figure(figsize=(15, 8))
plot_tree(tree, filled=True, feature_names=['Feature 1', 'Feature 2'], 
         class_names=['Class 0', 'Class 1'], fontsize=10)
plt.title('Decision Tree Visualization')
plt.show()

## 6.3 Random Forest

### Theory

**Random Forest:** Ensemble of decision trees

**Bagging (Bootstrap Aggregating):**
1. Create B bootstrap samples
2. Train tree on each sample
3. Average predictions (regression) or vote (classification)

**Random Forest additions:**
- Each split: consider random subset of features
- Decorrelates trees
- Reduces variance

**Advantages:**
- High accuracy
- Robust to overfitting
- Feature importance
- Handles missing data

**Hyperparameters:**
- n_estimators: number of trees
- max_depth: tree depth
- max_features: features per split

In [ ]:
# Random Forest example
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

rf_train_acc = rf.score(X_train, y_train)
rf_test_acc = rf.score(X_test, y_test)

print(f'Random Forest Results:')
print(f'Train Accuracy: {rf_train_acc:.3f}')
print(f'Test Accuracy: {rf_test_acc:.3f}')

# Compare single tree vs forest
models = {
    'Single Tree': tree,
    'Random Forest (100 trees)': rf
}

print('\nComparison:')
for name, model in models.items():
    train = model.score(X_train, y_train)
    test = model.score(X_test, y_test)
    print(f'{name:25s} Train: {train:.3f}  Test: {test:.3f}  Gap: {train-test:.3f}')

# Feature importance
importances = rf.feature_importances_
print(f'\nFeature Importances:')
for i, imp in enumerate(importances):
    print(f'Feature {i+1}: {imp:.3f}')

plt.figure(figsize=(10, 5))
plt.bar(['Feature 1', 'Feature 2'], importances, edgecolor='black', alpha=0.7)
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance')
plt.show()

## Summary: Machine Learning Key Concepts

### Algorithm Selection

| Algorithm | Pros | Cons | Use When |
|-----------|------|------|----------|
| KNN | Simple, no training | Slow prediction, scaling needed | Small datasets, baseline |
| Decision Tree | Interpretable, fast | Overfits, high variance | Need interpretability |
| Random Forest | Accurate, robust | Black box, slow | High accuracy needed |
| Boosting | Best accuracy | Complex, slow training | Competitions, max performance |

### Best Practices
- Always split train/test
- Use cross-validation
- Scale features for distance-based methods
- Start simple, add complexity if needed
- Monitor train vs test gap (overfitting)
- Tune hyperparameters systematically